In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("ggplot")

In [ ]:
DATA_PATH = Path("../../data/egx")

BUY_DROP = -0.05       # Buy if weekly return <= -5%
SELL_GAIN = 0.10       # Sell if weekly return >= 10%

BUY_AMOUNT = 5         # $
SELL_AMOUNT = 10       # $

INITIAL_CASH = 10000

In [ ]:
files = list(DATA_PATH.glob("*.csv"))

print("Stocks:", len(files))

sample = pd.read_csv(files[0])

sample.head()

In [ ]:
from pathlib import Path

DATA_PATH = Path("../../data/egx")

files = list(DATA_PATH.glob("*.csv"))

print(len(files))
print(files[:5])

In [ ]:
prices = pd.DataFrame()

for file in files:

    symbol = file.stem

    df = pd.read_csv(file)

    df["date"] = pd.to_datetime(df["date"])

    df = df.sort_values("date")

    df.set_index("date", inplace=True)

    # آخر سعر إغلاق كل أسبوع
    weekly = df["close"].resample("W-FRI").last()

    prices[symbol] = weekly

prices = prices.sort_index()

# ملء القيم المفقودة
prices = prices.ffill()

prices.head()

In [ ]:
returns = prices.pct_change()

returns.head()

In [ ]:
cash = INITIAL_CASH

positions = pd.Series(0.0, index=prices.columns)

portfolio_values = []
portfolio_dates = []

transactions = []

buy_counts = []
sell_counts = []

for date in prices.index[1:]:

    week_return = returns.loc[date]
    week_price = prices.loc[date]

    buys = 0
    sells = 0

    # ===== BUY =====
    for stock in prices.columns:

        if pd.isna(week_return[stock]):
            continue

        if week_return[stock] <= BUY_DROP:

            qty = BUY_AMOUNT / week_price[stock]

            positions[stock] += qty

            cash -= BUY_AMOUNT

            buys += 1

            transactions.append([
                date,
                stock,
                "BUY",
                week_price[stock],
                qty,
                BUY_AMOUNT
            ])

    # ===== SELL =====
    for stock in prices.columns:

        if pd.isna(week_return[stock]):
            continue

        if week_return[stock] >= SELL_GAIN:

            qty = SELL_AMOUNT / week_price[stock]

            qty = min(qty, positions[stock])

            if qty > 0:

                positions[stock] -= qty

                cash += qty * week_price[stock]

                sells += 1

                transactions.append([
                    date,
                    stock,
                    "SELL",
                    week_price[stock],
                    qty,
                    qty * week_price[stock]
                ])

    portfolio = cash + (positions * week_price).sum()

    portfolio_values.append(portfolio)
    portfolio_dates.append(date)

    buy_counts.append(buys)
    sell_counts.append(sells)

In [ ]:
portfolio = pd.Series(
    portfolio_values,
    index=portfolio_dates,
    name="Portfolio Value"
)

transactions = pd.DataFrame(
    transactions,
    columns=[
        "Date",
        "Symbol",
        "Action",
        "Price",
        "Quantity",
        "Amount"
    ]
)

portfolio.head()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(portfolio.index,
         portfolio.values,
         linewidth=2)

plt.title("Portfolio Value")

plt.xlabel("Date")

plt.ylabel("Portfolio ($)")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(portfolio.index,
         buy_counts,
         label="Buy")

plt.plot(portfolio.index,
         sell_counts,
         label="Sell")

plt.legend()

plt.title("Weekly Buy / Sell Signals")

plt.show()

In [ ]:
weekly_returns = portfolio.pct_change()

plt.figure(figsize=(10,5))

plt.hist(
    weekly_returns.dropna(),
    bins=30
)

plt.title("Weekly Portfolio Returns")

plt.show()

In [ ]:
def plot_stock_strategy(symbol):

    df = pd.read_csv(DATA_PATH / f"{symbol}.csv")

    df["date"] = pd.to_datetime(df["date"])

    df = df.sort_values("date")

    df.set_index("date", inplace=True)

    weekly = df["close"].resample("W-FRI").last().dropna()

    returns = weekly.pct_change()

    buy_dates = weekly[returns <= BUY_DROP]
    sell_dates = weekly[returns >= SELL_GAIN]

    plt.figure(figsize=(16,6))

    plt.plot(
        weekly.index,
        weekly.values,
        label="Close Price",
        linewidth=2
    )

    plt.scatter(
        buy_dates.index,
        buy_dates.values,
        marker="^",
        s=120,
        label="BUY"
    )

    plt.scatter(
        sell_dates.index,
        sell_dates.values,
        marker="v",
        s=120,
        label="SELL"
    )

    plt.title(symbol)

    plt.legend()

    plt.grid(True)

    plt.show()

In [ ]:
plot_stock_strategy("COMI")

In [ ]:
holding_history = []

holding_history.append(
    (positions * week_price).copy()
)

holding_history = pd.DataFrame(
    holding_history,
    index=portfolio_dates
)

plt.figure(figsize=(15,6))

holding_history.plot.area(
    stacked=True,
    figsize=(15,6)
)

plt.title("Portfolio Composition")

plt.show()

In [ ]:
def total_return(portfolio):

    return (
        portfolio.iloc[-1] /
        portfolio.iloc[0]
    ) - 1

print(
    f"Total Return : {total_return(portfolio):.2%}"
)

In [ ]:
def max_drawdown(portfolio):

    peak = portfolio.cummax()

    dd = (
        portfolio - peak
    ) / peak

    return dd.min()

print(
    f"Max Drawdown : {max_drawdown(portfolio):.2%}"
)

In [ ]:
running_max = portfolio.cummax()

drawdown = (
    portfolio-running_max
)/running_max

fig,(ax1,ax2)=plt.subplots(
    2,
    1,
    figsize=(14,7),
    sharex=True
)

ax1.plot(
    portfolio.index,
    portfolio
)

ax1.set_title(
    "Portfolio Value"
)

ax2.fill_between(
    drawdown.index,
    drawdown,
    0
)

ax2.set_title(
    "Drawdown"
)

plt.show()

In [ ]:
rolling = (
    portfolio
    .pct_change(52)
)*100

plt.figure(figsize=(15,5))

plt.plot(
    rolling
)

plt.title(
    "Rolling 1-Year Return"
)

plt.show()

In [ ]:
vol = (
    portfolio
    .pct_change()
    .rolling(52)
    .std()
)*np.sqrt(52)

plt.figure(figsize=(15,5))

plt.plot(vol)

plt.title(
    "Rolling Volatility"
)

plt.show()

In [ ]:
r = portfolio.pct_change()

rolling_sharpe = (
    r.rolling(52).mean()
    /
    r.rolling(52).std()
)*np.sqrt(52)

plt.figure(figsize=(15,5))

plt.plot(
    rolling_sharpe
)

plt.title(
    "Rolling Sharpe Ratio"
)

plt.show()

In [ ]:
buy_signals = (returns <= BUY_DROP).sum()

buy_signals = buy_signals.sort_values(ascending=False)

buy_signals.head(10)

In [ ]:
plt.figure(figsize=(10,6))

buy_signals.head(10).plot.bar()

plt.title("Top 10 Buy Signals")

plt.show()

In [ ]:
sell_signals = (returns >= SELL_GAIN).sum()

sell_signals = sell_signals.sort_values(ascending=False)

sell_signals.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sell_signals.head(10).plot.bar()

plt.title("Top 10 Sell Signals")

plt.show()

In [ ]:
stock_returns = (
    prices.iloc[-1] / prices.iloc[0] - 1
) * 100

stock_returns = stock_returns.sort_values(ascending=False)

stock_returns.head(10)

In [ ]:
plt.figure(figsize=(12,6))

stock_returns.head(10).plot.bar()

plt.title("Top Performing Stocks")

plt.ylabel("% Return")

plt.show()

In [ ]:
portfolio.to_csv("portfolio_history.csv")

transactions.to_csv("transactions.csv", index=False)

positions.to_csv("holdings.csv")

returns.to_csv("weekly_returns.csv")

buy_signals.to_csv("buy_signals.csv")

sell_signals.to_csv("sell_signals.csv")

In [ ]:
portfolio_returns = portfolio.pct_change().dropna()

total_return = (portfolio.iloc[-1] - INITIAL_CASH) / INITIAL_CASH

years = (portfolio.index[-1] - portfolio.index[0]).days / 365.25

cagr = ((portfolio.iloc[-1] / INITIAL_CASH) ** (1 / years)) - 1

sharpe = (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(52)

running_max = portfolio.cummax()

drawdown = (portfolio - running_max) / running_max

max_drawdown = drawdown.min()

performance = pd.DataFrame({
    "Metric": [
        "Initial Cash",
        "Final Portfolio",
        "Total Return %",
        "CAGR %",
        "Sharpe Ratio",
        "Max Drawdown %"
    ],
    "Value": [
        INITIAL_CASH,
        portfolio.iloc[-1],
        total_return * 100,
        cagr * 100,
        sharpe,
        max_drawdown * 100
    ]
})

performance

In [ ]:
performance.to_csv("performance.csv", index=False)

In [ ]:
last_prices = prices.iloc[-1]

holdings = pd.DataFrame({
    "Symbol": positions.index,
    "Shares": positions.values,
    "Last Price": last_prices.values
})

holdings["Market Value"] = holdings["Shares"] * holdings["Last Price"]

holdings = holdings[holdings["Shares"] > 0]

holdings.sort_values("Market Value", ascending=False)

In [ ]:
holdings.to_csv("holdings.csv", index=False)